## **Question 2a:Arithmetic and Geometric progression in population projections (the group chose D02)**
##### **a. For the selected projection district, fit arithmetic and geometric growth using opening counts on 1**
##### **January 2020 and 1 January 2025. Project the total to 1 January 2030 under each model. Identify which**
##### **inputs are estimates and which results are conditional projections**

**1.Loading the required datasets and also having a view of the datasets**


In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load datasets
districts = pd.read_csv("../data/districts.csv")
age_sex = pd.read_csv("../data/age_sex.csv")
scenarios = pd.read_csv("../data/projection_scenarios.csv")

# Display datasets
display(districts.head())
display(age_sex.head())
display(scenarios)

,district_id,district_name,region,population_2020,population_2026,births_2025,deaths_2025,arrivals_2025,departures_2025,piped_water_pct
0,D01,District D01,Region 1,156609,173752,3939,2166,2460,2926,31.3
1,D02,District D02,Region 1,103634,114605,2480,1173,2705,1618,80.5
2,D03,District D03,Region 1,72900,77096,1008,1639,1278,1458,56.6
3,D04,District D04,Region 2,83366,96331,2268,1239,1303,1655,NaN
4,D05,District D05,Region 2,137858,156477,3469,1597,3910,1749,77.4


,district_id,age_start,sex,population_2025,baseline_deaths_2025,arrivals_2025,departures_2025,births_to_women_2025,survival_5yr
0,D01,0,F,9891,56,116,172,0,0.972182
1,D01,0,M,9031,52,119,133,0,0.971438
2,D01,5,F,9029,6,90,117,0,0.996498
3,D01,5,M,7501,7,142,164,0,0.995145
4,D01,10,F,8655,5,74,112,0,0.997013


,scenario,fertility_multiplier,arrivals_multiplier,departures_multiplier,newborn_survival_f,newborn_survival_m
0,Low,0.88,0.82,1.12,0.970,0.963
1,Base,1.00,1.00,1.00,0.978,0.970
2,High,1.12,1.18,0.88,0.986,0.980


#### **2 Extracting the population totals**

In [62]:
# Population on 1 January 2020
population_2020 = districts["population_2020"].iloc[1]

# Population on 1 January 2025
#population_2025 = age_sex["population_2025"].sum()
row = districts.iloc[1]

# Reconstruct 2025 opening population
population_2025 = (
    row["population_2026"]
    - row["births_2025"]
    + row["deaths_2025"]
    - row["arrivals_2025"]
    + row["departures_2025"]
)

# Reported population on 1 January 2026
population_2026 = districts["population_2026"].iloc[1]

print(f"Population on 1 January 2020: {population_2020:,.0f}")
print(f"Population on 1 January 2025: {population_2025:,.0f}")
print(f"Reported population on 1 January 2026: {population_2026:,.0f}")

Population on 1 January 2020: 103,634
Population on 1 January 2025: 112,211
Reported population on 1 January 2026: 114,605


##### **Calculate arithmetic and geometric growth rates**

In [63]:
# Calculate arithmetic annual growth rate
arithmetic_growth = (
    (population_2025 - population_2020)
    / (5)
)

# Calculate geometric annual growth rate
geometric_growth = (
    (population_2025 / population_2020) ** (1/5)
) - 1

print(f"Arithmetic annual growth rate: {arithmetic_growth:.4f}")
print(f"Geometric annual growth rate: {geometric_growth * 100:.4f}%")

Arithmetic annual growth rate: 1715.4000
Geometric annual growth rate: 1.6030%


##### **Projected populations as in 2030**

In [64]:
# Project to 2030
arithmetic_2030 = population_2025 + 5*(arithmetic_growth)

geometric_2030 = population_2025 * (1 + geometric_growth) ** 5

# Present results
projection_comparison = pd.DataFrame({
    "Method": ["Arithmetic", "Geometric"],
    "Annual Growth Rate(%) ": [
        arithmetic_growth ,
        geometric_growth * 100
    ],
    "Projected Population 2030": [
        round(arithmetic_2030),
        round(geometric_2030)
    ]
})

display(projection_comparison)

,Method,Annual Growth Rate(%),Projected Population 2030
0,Arithmetic,1715.400000,120788
1,Geometric,1.603024,121498


### **COMMENT**
From the above we can see that the population projections using the arithmetic vs the geometric methods are not  equal.This reveals that arithmetic accummulates a fixed population to the previous each year vs geometric which accummulates a certain percentage giving rise to a higher projection

# **Part 2:Cohort Based Projections**
b. Build a five-year cohort-component projection for the same date under all three scenarios. Advance
each starting cohort using its supplied survival probability. The 85+ group receives survivors from ages
80–84 and survivors already aged 85+. Estimate births as five times 2025 births to women, multiplied by
the scenario fertility factor. Assume 49% female and 51% male births, apply the scenario newborn survival
rates, and place surviving births in ages 0–4. Apply five times each 2025 migration flow, scaled by its
scenario multiplier, as an end-of-period adjustment by destination age and sex, including ages 0–4.
Document the timing assumption and verify the accounting totals.*

##### **Making sure that we selected the correct district which is district 2**

In [65]:
# Filter the age-sex dataset for D02 only
projection_data = age_sex[
    age_sex["district_id"] == "D02"
].copy()

# Sort the records
projection_data = projection_data.sort_values(
    ["age_start", "sex"]
).reset_index(drop=True)

# Verify selection
print("Selected district:", projection_data["district_id"].unique())

print("Number of records:", len(projection_data))

print("Number of age groups:",
      projection_data["age_start"].nunique())

print("Sex categories:",
      projection_data["sex"].unique())

print("\nTotal 2025 population:",
      f"{projection_data['population_2025'].sum():,.0f}")

Selected district: <ArrowStringArray>
['D02']
Length: 1, dtype: str
Number of records: 36
Number of age groups: 18
Sex categories: <ArrowStringArray>
['F', 'M']
Length: 2, dtype: str

Total 2025 population: 112,211


##### **02:Create the survival projection**

In [66]:
# Create readable five-year age-group labels
def age_group_label(age):
    if age == 85:
        return "85+"
    return f"{age}-{age + 4}"

projection_data["age_group"] = (
    projection_data["age_start"].apply(age_group_label)
)

display(
    projection_data[
        ["age_start", "age_group", "sex", "population_2025"]
    ]
)

,age_start,age_group,sex,population_2025
0,0,0-4,F,2030
1,0,0-4,M,1787
2,5,5-9,F,2498
3,5,5-9,M,2540
4,10,10-14,F,3049
5,10,10-14,M,3012
6,15,15-19,F,3541
7,15,15-19,M,3095
8,20,20-24,F,4214
9,20,20-24,M,4042


In [67]:
# Create an empty dataframe for projected survivors
survivors = projection_data[
    ["age_start", "sex","age_group"]
].copy()

survivors["survivors_2030"] = 0.0

# Project each cohort into its next age group
for sex in ["F", "M"]:

    sex_data = projection_data[
        projection_data["sex"] == sex
    ]

    for _, row in sex_data.iterrows():

        age = row["age_start"]

        population = row["population_2025"]

        survival = row["survival_5yr"]

        surviving_population = population * survival

        # Ages below 85 move into the next age group
        if age < 85:

            next_age = age + 5

        # The 85+ group remains in 85+
        else:

            next_age = 85

        # Add survivors to the destination age group
        survivors.loc[
            (survivors["age_start"] == next_age) &
            (survivors["sex"] == sex),
            "survivors_2030"
        ] += surviving_population


display(survivors)

,age_start,sex,age_group,survivors_2030
0,0,F,0-4,0.000000
1,0,M,0-4,0.000000
2,5,F,5-9,1981.562170
3,5,M,5-9,1721.254483
4,10,F,10-14,2489.564254
5,10,M,10-14,2527.421920
6,15,F,15-19,3041.347010
7,15,M,15-19,3002.876652
8,20,F,20-24,3523.248967
9,20,M,20-24,3078.271525


##### **Investigate the birth data**

In [68]:
# Examine births by female age group
female_birth_data = projection_data[
    projection_data["sex"] == "F"
][["age_start", "age_group", "births_to_women_2025"]]

display(female_birth_data)

print("Total births by age group:",
      female_birth_data["births_to_women_2025"].sum())

#print("District births:",
      #district_births)

,age_start,age_group,births_to_women_2025
0,0,0-4,0
2,5,5-9,0
4,10,10-14,0
6,15,15-19,132
8,20,20-24,451
10,25,25-29,684
12,30,30-34,646
14,35,35-39,384
16,40,40-44,147
18,45,45-49,36


Total births by age group: 2480


##### **Calculating the births in the five years**

In [69]:
# Verified annual births
annual_births = female_birth_data["births_to_women_2025"].sum()

# Calculate five-year births under each scenario
birth_results = scenarios[[
    "scenario",
    "fertility_multiplier",
    "newborn_survival_f",
    "newborn_survival_m"
]].copy()

birth_results["total_births_5yr"] = (
    5 * annual_births * birth_results["fertility_multiplier"]
)

# Split births by sex
birth_results["female_births"] = (
    birth_results["total_births_5yr"] * 0.49
)

birth_results["male_births"] = (
    birth_results["total_births_5yr"] * 0.51
)

display(birth_results)

,scenario,fertility_multiplier,newborn_survival_f,newborn_survival_m,total_births_5yr,female_births,male_births
0,Low,0.88,0.970,0.963,10912.0,5346.88,5565.12
1,Base,1.00,0.978,0.970,12400.0,6076.00,6324.00
2,High,1.12,0.986,0.980,13888.0,6805.12,7082.88


##### **Adding the new born survival**

In [70]:
birth_results["surviving_female_newborns"] = (
    birth_results["female_births"] *
    birth_results["newborn_survival_f"]
)

birth_results["surviving_male_newborns"] = (
    birth_results["male_births"] *
    birth_results["newborn_survival_m"]
)

birth_results["total_surviving_newborns"] = (
    birth_results["surviving_female_newborns"] +
    birth_results["surviving_male_newborns"]
)

display(birth_results[[
    "scenario",
    "total_births_5yr",
    "surviving_female_newborns",
    "surviving_male_newborns",
    "total_surviving_newborns"
]])

,scenario,total_births_5yr,surviving_female_newborns,surviving_male_newborns,total_surviving_newborns
0,Low,10912.0,5186.47360,5359.21056,10545.68416
1,Base,12400.0,5942.32800,6134.28000,12076.60800
2,High,13888.0,6709.84832,6941.22240,13651.07072


In [71]:
#Seperating the new borns and the survival
# Create a dictionary to store newborn populations
newborns = {}

for _, row in birth_results.iterrows():

    scenario = row["scenario"]

    newborns[(scenario, "F")] = row["surviving_female_newborns"]
    newborns[(scenario, "M")] = row["surviving_male_newborns"]


# Display newborn population by scenario and sex
print("Surviving newborns by scenario:")

for key, value in newborns.items():
    print(key, ":", round(value, 2))

Surviving newborns by scenario:
('Low', 'F') : 5186.47
('Low', 'M') : 5359.21
('Base', 'F') : 5942.33
('Base', 'M') : 6134.28
('High', 'F') : 6709.85
('High', 'M') : 6941.22


Calculating arrivals and departures

In [72]:
# Calculate migration under each scenario

migration_results = []

for _, scenario in scenarios.iterrows():

    scenario_name = scenario["scenario"]

    temp = projection_data[
        ["age_start", "age_group", "sex",
         "arrivals_2025", "departures_2025"]
    ].copy()

    temp["scenario"] = scenario_name

    temp["arrivals_5yr"] = (
        temp["arrivals_2025"] * 5 *
        scenario["arrivals_multiplier"]
    )

    temp["departures_5yr"] = (
        temp["departures_2025"] * 5 *
        scenario["departures_multiplier"]
    )

    migration_results.append(temp)

migration_results = pd.concat(
    migration_results,
    ignore_index=True
)

display(migration_results.head(15))

,age_start,age_group,sex,arrivals_2025,departures_2025,scenario,arrivals_5yr,departures_5yr
0,0,0-4,F,39,30,Low,159.9,168.0
1,0,0-4,M,39,20,Low,159.9,112.0
2,5,5-9,F,41,29,Low,168.1,162.4
3,5,5-9,M,44,23,Low,180.4,128.8
4,10,10-14,F,61,53,Low,250.1,296.8
5,10,10-14,M,45,50,Low,184.5,280.0
6,15,15-19,F,81,44,Low,332.1,246.4
7,15,15-19,M,102,46,Low,418.2,257.6
8,20,20-24,F,77,52,Low,315.7,291.2
9,20,20-24,M,80,72,Low,328.0,403.2


Calculating the final 2030 population for district 02

In [73]:
# Store the final projections for all scenarios
projection_results = []

for scenario in ["Low", "Base", "High"]:

    # Start with existing population survivors
    temp = survivors.copy()

    # Add migration for the selected scenario
    migration_temp = migration_results[
        migration_results["scenario"] == scenario
    ]

    temp = temp.merge(
        migration_temp[
            ["age_start", "sex", "arrivals_5yr", "departures_5yr"]
        ],
        on=["age_start", "sex"],
        how="left"
    )

    # Add surviving newborns to the 0-4 age group
    temp["newborns_2030"] = 0.0

    temp.loc[
        (temp["age_start"] == 0) & (temp["sex"] == "F"),
        "newborns_2030"
    ] = newborns[(scenario, "F")]

    temp.loc[
        (temp["age_start"] == 0) & (temp["sex"] == "M"),
        "newborns_2030"
    ] = newborns[(scenario, "M")]

    # Calculate final population
    temp["population_2030"] = (
        temp["survivors_2030"]
        + temp["newborns_2030"]
        + temp["arrivals_5yr"]
        - temp["departures_5yr"]
    )

    temp["scenario"] = scenario

    projection_results.append(temp)

# Combine all scenarios
projection_results = pd.concat(
    projection_results,
    ignore_index=True
)

display(projection_results)

,age_start,sex,age_group,survivors_2030,arrivals_5yr,departures_5yr,newborns_2030,population_2030,scenario
0,0,F,0-4,0.000000,159.9,168.0,5186.47360,5178.373600,Low
1,0,M,0-4,0.000000,159.9,112.0,5359.21056,5407.110560,Low
2,5,F,5-9,1981.562170,168.1,162.4,0.00000,1987.262170,Low
3,5,M,5-9,1721.254483,180.4,128.8,0.00000,1772.854483,Low
4,10,F,10-14,2489.564254,250.1,296.8,0.00000,2442.864254,Low
...,...,...,...,...,...,...,...,...,...
103,75,M,75-79,1255.083776,218.3,61.6,0.00000,1411.783776,High
104,80,F,80-84,1064.807900,112.1,52.8,0.00000,1124.107900,High
105,80,M,80-84,997.244424,100.3,52.8,0.00000,1044.744424,High
106,85,F,85+,1108.065231,129.8,30.8,0.00000,1207.065231,High


In [74]:
#The totals
scenario_totals = (
    projection_results
    .groupby("scenario")["population_2030"]
    .sum()
    .reset_index()
    .round(0)
)

scenario_totals.columns = [
    "Scenario",
    "Projected_Population_2030"
]

display(scenario_totals)

,Scenario,Projected_Population_2030
0,Base,124570.0
1,High,129550.0
2,Low,119634.0


Performing the accounting checks

In [75]:
# Calculate accounting components for each scenario

accounting_checks = []

for scenario in ["Low", "Base", "High"]:

    temp = projection_results[
        projection_results["scenario"] == scenario
    ]

    initial_population = projection_data["population_2025"].sum()

    survivors_total = temp["survivors_2030"].sum()

    newborns_total = temp["newborns_2030"].sum()

    arrivals_total = temp["arrivals_5yr"].sum()

    departures_total = temp["departures_5yr"].sum()

    projected_total = temp["population_2030"].sum()

    calculated_total = (
        survivors_total
        + newborns_total
        + arrivals_total
        - departures_total
    )

    accounting_checks.append({
        "Scenario": scenario,
        "Initial_Population": initial_population,
        "Survivors": survivors_total,
        "Newborns": newborns_total,
        "Arrivals": arrivals_total,
        "Departures": departures_total,
        "Projected_Population": projected_total,
        "Accounting_Difference": projected_total - calculated_total
    })

accounting_checks = pd.DataFrame(accounting_checks)

display(accounting_checks.round(0))

,Scenario,Initial_Population,Survivors,Newborns,Arrivals,Departures,Projected_Population,Accounting_Difference
0,Low,112211,107058.0,10546.0,11090.0,9061.0,119634.0,-0.0
1,Base,112211,107058.0,12077.0,13525.0,8090.0,124570.0,0.0
2,High,112211,107058.0,13651.0,15959.0,7119.0,129550.0,0.0


##### **Interpretation**
From the above we can see different scenarios being applied to the projections of the 2030 populations .The different scenarios give rise to different population projections in the year 2030 .With each scenario giving aa different size by the year 2030.This different scenarios can be used in different planning cases by the authorities and companies .For examle if the government want to build a water supply system that sustains everyone by the year 2030 it will then use the high scenario so that it will build a system that isnt suffocated by the then available population but a system that is rather way more caacitated than the population in that year.So the use of either projection depends on the type of decision to be made based on the projections

## **Part C:Comparing**
c. Compare the cohort-component totals with the arithmetic and geometric results and report the projected
shares under 15 and aged 65+. From the Base scenario, change only its fertility multiplier to the High
value. Quantify the effect on ages 0–4 and explain one planning implication and one limitation.

##### **Creating the comparison table**

In [76]:
# Compare the three projection methods

comparison = pd.DataFrame({
    "Method": [
        "Arithmetic",
        "Geometric",
        "Cohort-component (Low)",
        "Cohort-component (Base)",
        "Cohort-component (High)"
    ],
    "Projected_Population_2030": [
        arithmetic_2030,
        geometric_2030,
        scenario_totals.loc[
            scenario_totals["Scenario"] == "Low",
            "Projected_Population_2030"
        ].iloc[0],
        scenario_totals.loc[
            scenario_totals["Scenario"] == "Base",
            "Projected_Population_2030"
        ].iloc[0],
        scenario_totals.loc[
            scenario_totals["Scenario"] == "High",
            "Projected_Population_2030"
        ].iloc[0]
    ]
})

display(comparison.round(0))

,Method,Projected_Population_2030
0,Arithmetic,120788.0
1,Geometric,121498.0
2,Cohort-component (Low),119634.0
3,Cohort-component (Base),124570.0
4,Cohort-component (High),129550.0


##### **Population Shares by Age**

In [77]:
# Calculate population shares by age group

age_shares = projection_results.copy()

# Identify age categories
age_shares["under_15"] = age_shares["age_start"] < 15

age_shares["age_65_plus"] = age_shares["age_start"] >= 65

# Aggregate by scenario
age_summary = age_shares.groupby("scenario").apply(
    lambda x: pd.Series({
        "Total_Population": x["population_2030"].sum(),

        "Population_Under_15": x.loc[
            x["under_15"], "population_2030"
        ].sum(),

        "Population_65_Plus": x.loc[
            x["age_65_plus"], "population_2030"
        ].sum()
    }),
    include_groups=False
).reset_index()

# Calculate percentages
age_summary["Under_15_Percent"] = (
    age_summary["Population_Under_15"] /
    age_summary["Total_Population"] * 100
)

age_summary["Age_65_Plus_Percent"] = (
    age_summary["Population_65_Plus"] /
    age_summary["Total_Population"] * 100
)

display(age_summary.round(2))

,scenario,Total_Population,Population_Under_15,Population_65_Plus,Under_15_Percent,Age_65_Plus_Percent
0,Base,124569.96,21116.41,17359.61,16.95,13.94
1,High,129549.72,23055.97,17803.31,17.80,13.74
2,Low,119633.74,19220.39,16915.91,16.07,14.14


##### **Fertility Sensitivity Analysis**

In [78]:
# Calculate newborns under the sensitivity scenario

base_female_births = 5 * annual_births * 1.12 * 0.49

base_male_births = 5 * annual_births * 1.12 * 0.51

# Apply BASE newborn survival probabilities
base_female_survivors = (
    base_female_births * 0.978
)

base_male_survivors = (
    base_male_births * 0.970
)

# Total surviving newborns
sensitivity_newborns = (
    base_female_survivors +
    base_male_survivors
)

print("Base newborns:", 
      round(birth_results.loc[
          birth_results["scenario"] == "Base",
          "total_surviving_newborns"
      ].iloc[0], 2))

print("Sensitivity newborns:",
      round(sensitivity_newborns, 2))

Base newborns: 12076.61
Sensitivity newborns: 13525.8


##### **Effect on the 0–4 age group**

In [79]:
# Get the original Base scenario's 0–4 population
base_0_4 = projection_results.loc[
    (projection_results["scenario"] == "Base") &
    (projection_results["age_start"] == 0),
    "population_2030"
].sum()

# Get original Base surviving newborns
base_newborns = birth_results.loc[
    birth_results["scenario"] == "Base",
    "total_surviving_newborns"
].iloc[0]

# Calculate the revised 0–4 population
sensitivity_0_4 = (
    base_0_4 - base_newborns + sensitivity_newborns
)

# Calculate the absolute and percentage changes
absolute_change = sensitivity_0_4 - base_0_4
percentage_change = (absolute_change / base_0_4) * 100

print("Base 0–4 population:", round(base_0_4, 0))
print("Sensitivity 0–4 population:", round(sensitivity_0_4, 0))
print("Absolute increase:", round(absolute_change, 0))
print("Percentage increase:", round(percentage_change, 2), "%")

Base 0–4 population: 12217.0
Sensitivity 0–4 population: 13666.0
Absolute increase: 1449.0
Percentage increase: 11.86 %


##### **Interpretation**
A fertility sensitivity analysis was conducted by increasing the Base scenario fertility multiplier from 1.00 to 1.12 while keeping all other Base assumptions unchanged. The projected population aged 0–4 increased from 12,217 to 13,666 by 2030, representing an increase of approximately 1,449 people or 11.86%. This demonstrates that fertility assumptions have a considerable influence on projected demand for early childhood healthcare, immunisation and educational facilities. However, the results are conditional on the assumed survival and migration patterns remaining unchanged.



In [80]:
projection_comparison.to_csv('../outputs/projections/projection_comparison.csv')
comparison.to_csv('../outputs/projections/whole_comparison.csv')

